# GLiNER2 Vertical LoRA — Colab Training Driver

Trains a PEFT LoRA adapter for a hacienda vertical **entirely on Colab** and exports it in
the exact shape `xberg-gliner`'s merge-at-load path accepts. Nothing here runs locally.

Spec: `superpowers/specs/2026-08-15-gliner2-lora-training-driver-plan.md`
Plan: `superpowers/plans/2026-08-15-gliner2-lora-training-driver-implementation.md`

### Before you start

1. **Runtime → Change runtime type → T4 GPU.** §6 trains; that is what the GPU is for.
2. **Mount Drive** (§0). Colab disconnects at ~90 min idle and hard-caps at 12 h. A run that
   writes only to `/content` loses the adapter *and* forces a 1.2 GB re-download next
   session. Everything durable goes to Drive.
3. **Run §1–§5 first.** They cost no GPU quota — a few kilobytes and under a minute on any
   runtime — and they catch every known way this fails. GPU quota is the scarce resource on
   Colab; spending a session to discover a malformed corpus is the thing to avoid.

### Why the notebook is shaped this way

Five GLiNER2 defaults produce an adapter the runtime **rejects**, and two more silently
destroy training signal with no error at all. Each is pinned explicitly below, with the
reason inline. Verified against GLiNER2 `main` and `xberg` rev `9bfbc10`.

| Hazard | Default | Pinned | Consequence if left alone |
|---|---|---|---|
| `fp16` | `True` | `False` | adapter saved F16; `lora.rs:127` hard-rejects non-F32 |
| `bf16` | `False` | `False` | same dtype constraint |
| `lora_use_dora` | `False` | `False` | DoRA emits `lora_magnitude_vector`; parser rejects |
| `base_model_name_or_path` | HF slug | deploy basename | fails the substring guard, refuses to merge |
| `synthetic_entity_label_prob` | `0.2` | `0.0` | relabels 20% of types to "entity 1"/"entity 2" |
| base checkpoint | — | **F32** | `decode_view` decodes F32/I64 only; an F16 base cannot be merged into |
| `max_width` | `8` tokens | measured | wider spans silently unlearnable, no error |
| `sanitize()` | — | measured | one unfound mention drops the **whole entity type** for that record |

Note the first two rows: the adapter must end up F32, so mixed precision is off. Training is
full-precision on the GPU — still far faster than CPU, just not as fast as an fp16 run.

## 0. Session setup — Drive, install, device

In [ ]:
%pip install -q peft safetensors

import torch, peft
print('torch', torch.__version__, '| peft', peft.__version__)

GPU = torch.cuda.is_available()
print('device:', torch.cuda.get_device_name(0) if GPU else 'CPU')
if not GPU:
    print('\nNo GPU. Sections 1-5 are unaffected -- run them now, they cost no quota.')
    print('Before section 6: Runtime > Change runtime type > T4 GPU.')

In [ ]:
BASE_REPO = 'fastino/GLiNER2-Guardrails-PII-Multi'   # upstream is all-F32 (verified)
DEPLOY_BASENAME = 'gliner2'                          # basename of the DEPLOYED model dir
VERTICAL_ID = 'business_law'

USE_DRIVE = True
DRIVE_ROOT = '/content/drive/MyDrive/hacienda-lora'

LORA_R = 16
LORA_ALPHA = 32.0        # merge scale is alpha/r (lora.rs:214); no rslora support
NUM_EPOCHS = 3
BATCH_SIZE = 2
ENCODER_LR = 1e-5
TASK_LR = 5e-4
MAX_WIDTH = 8            # GLiNER2 span width cap, in TOKENS

In [ ]:
import pathlib

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = pathlib.Path(DRIVE_ROOT) / VERTICAL_ID
else:
    # Ephemeral: a disconnect loses the adapter and forces a 1.2 GB re-download.
    WORK = pathlib.Path('/content') / VERTICAL_ID

WORK.mkdir(parents=True, exist_ok=True)
OUT_DIR = str(WORK / 'train')       # trainer checkpoints
ADAPTER_DIR = str(WORK / 'adapter')  # final, loader-ready adapter
CORPUS = WORK / 'corpus.jsonl'       # put your JSONL here to skip the upload prompt
HF_CACHE = str(WORK / 'hf-cache') if USE_DRIVE else None

print('workdir:', WORK)
print('corpus :', CORPUS, '(exists)' if CORPUS.exists() else '(not present yet)')

## 1. Configuration note — the value that silently kills a finished run

`DEPLOY_BASENAME` is the most error-prone constant here. The load-time guard
(`candle/model.rs:137`) does a **bidirectional, case-sensitive substring** check between the
adapter's `base_model_name_or_path` and `model_id`, where `model_id` is the deployed model
**directory's basename** (`model.rs:199`) — not an HF slug. In this repo the model lives at
`apps/hacienda-studio/public/models/gliner2/`, so the basename is `gliner2`. Writing
`fastino/GLiNER2-Guardrails-PII-Multi` there is the classic failure: it neither contains nor
is contained by `gliner2`, so the merge is refused *after* training completes.

## 2. PEFT key-naming proof — no GPU, no download, no training

**This is the one assumption in the spec that was read from PEFT's source and never
executed.** Everything downstream depends on PEFT saving LoRA weights as exactly:

    base_model.model.<full base module path>.lora_{A,B}.weight

`parse_lora_key` (`lora.rs:172-185`) strips that prefix and suffix and nothing else, then
`merge_into_base` does an **exact HashMap lookup** of the remainder against each base key
minus `.weight` (`lora.rs:224-238`) — not a substring or suffix match. If PEFT's naming
differs by so much as a prefix, every adapter is refused (`lora.rs:246-253`).

That is a question about **PEFT's convention over a module tree**, not about GLiNER2's
weights — so it is answered here against a structural stub carrying GLiNER2's real module
path. One second, no quota.

**If this cell fails, stop.** Merge-at-load cannot work and no amount of GPU time fixes it.

In [ ]:
import tempfile
import torch.nn as nn
from peft import LoraConfig, get_peft_model
from safetensors import safe_open

# GLiNER2's real path for layer 0, read from the checkpoint header. The doubled prefix is
# not a typo: GLiNER2 nests HF's DebertaV2Model under its own `encoder` attribute, and
# DeBERTa-v2 names projections `*_proj`, not BERT's bare `query`/`key`/`value`.
REAL_PATH = 'encoder.encoder.layer.0.attention.self.query_proj'


def module_at(path):
    """Smallest nn.Module tree whose only Linear is reachable at exactly `path`."""
    parts = path.split('.')
    node = nn.Linear(8, 8, bias=False)
    while parts:
        name = parts.pop()
        if name.isdigit():
            # A numeric segment can only come from nn.ModuleList indexing, and PEFT sees
            # the index as a path component just as the checkpoint writer did.
            node = nn.ModuleList([nn.Identity() for _ in range(int(name))] + [node])
            name = parts.pop()
        parent = nn.Module()
        setattr(parent, name, node)
        node = parent
    return node


stub = module_at(REAL_PATH)
assert REAL_PATH in dict(stub.named_modules()), 'stub does not reproduce the real path'

peft_model = get_peft_model(stub, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.0,
    use_dora=False,                 # DoRA adds lora_magnitude_vector keys the parser rejects
    target_modules=[REAL_PATH],
))

with tempfile.TemporaryDirectory() as d:
    peft_model.save_pretrained(d)
    st = pathlib.Path(d) / 'adapter_model.safetensors'
    assert st.exists(), (
        f'peft {peft.__version__} did not write safetensors; got '
        f'{[p.name for p in pathlib.Path(d).iterdir()]}. The Rust loader reads '
        'adapter_model.safetensors only.'
    )
    with safe_open(str(st), framework='pt') as f:
        saved = sorted(f.keys())

expected = sorted(f'base_model.model.{REAL_PATH}.lora_{ab}.weight' for ab in ('A', 'B'))

print('saved by PEFT:')
for k in saved:
    print('  ', k)
print('\nrequired by parse_lora_key + merge_into_base:')
for k in expected:
    print('  ', k)

assert saved == expected, (
    'PEFT key naming does NOT match what the Rust loader parses. Every adapter produced '
    'here would be rejected at merge time. Fix the loader or the export, do not train.'
)
print(f'\nOK: peft {peft.__version__} names keys exactly as the loader expects. Pin this version.')

## 3. Base checkpoint contract — header only, ~2 KB

The safetensors header is a length-prefixed JSON blob at the start of the file, so a ranged
HTTP read gets every tensor name and dtype without pulling 1.2 GB. That is the whole
base-side contract, before committing a GPU session:

- **dtype is F32.** `decode_view` (`lora.rs:259-291`) decodes only F32 and I64, so an F16
  base cannot be merged into *at all*. The upstream checkpoint is all-F32; the F16 file at
  `apps/hacienda-studio/public/models/gliner2/model.safetensors` is the derived distribution
  artifact from `scripts/convert_gliner2_f16.py` and is **not** a merge base.
- **The real key names**, which are what the exact-match merge looks up.

In [ ]:
import json, struct, collections, requests
from huggingface_hub import hf_hub_url

url = hf_hub_url(BASE_REPO, 'model.safetensors')
n = struct.unpack('<Q', requests.get(url, headers={'Range': 'bytes=0-7'}).content)[0]
header = json.loads(requests.get(url, headers={'Range': f'bytes=8-{8 + n - 1}'}).content)
header.pop('__metadata__', None)

BASE_KEYS = sorted(header)
dtypes = collections.Counter(v['dtype'] for v in header.values())
print(f'read {n} header bytes | tensors: {len(BASE_KEYS)} | dtypes: {dict(dtypes)}')

assert set(dtypes) <= {'F32', 'I64'}, (
    f'base checkpoint has non-mergeable dtypes {dict(dtypes)}; use the upstream F32 '
    'checkpoint, NOT a converted F16 distribution artifact'
)
assert f'{REAL_PATH}.weight' in set(BASE_KEYS), (
    'the path proven in section 2 is not in the real checkpoint'
)

ATTN = [k[:-len('.weight')] for k in BASE_KEYS
        if '.attention.self.' in k
        and k.endswith(('query_proj.weight', 'key_proj.weight', 'value_proj.weight'))]
print('attention projections:', len(ATTN))
print('example:', ATTN[0] if ATTN else 'NONE FOUND -- check prefix assumptions')

# Fully-qualified fallback for section 6, if GLiNER2's group names target nothing
# mergeable. These are the paths the merge actually looks up.
FQ_TARGETS = ATTN

## 4. Training corpus

GLiNER2 takes **verbatim mention strings**, not word-index spans:

```json
{"input": "...", "output": {"entities": {"contracting_party": ["Acme Corp"]}}}
```

`{"tokenized_text", "ner"}` is GLiNER **v1** and is rejected by `_load_dict_list`. Produce
this shape with `dataset/assemble.py::to_gliner2_record`, which slices mentions out of the
chunk rather than rebuilding them from tokens — rebuilding drops punctuation and collapses
whitespace, which §5 then flags as unresolvable.

Drop your JSONL at the `CORPUS` path printed in §0 and it is picked up automatically; the
synthetic sample is only there to smoke-test the pipeline end to end.

In [ ]:
SYNTHETIC = [
    {'input': 'This Agreement is entered into by Acme Corp and Globex Inc.',
     'output': {'entities': {'contracting_party': ['Acme Corp', 'Globex Inc']}}},
    {'input': 'The Licensee shall not assign this Agreement without consent.',
     'output': {'entities': {'contracting_party': ['Licensee']}}},
    {'input': 'Neither party is liable for delay caused by force majeure.',
     'output': {'entities': {'force_majeure': ['force majeure']}}},
    {'input': 'Either party may terminate upon thirty days written notice.',
     'output': {'entities': {'termination_clause': ['terminate upon thirty days written notice']}}},
]

if CORPUS.exists():
    records = [json.loads(line) for line in CORPUS.read_text().splitlines() if line.strip()]
    print(f'loaded {len(records)} records from {CORPUS}')
else:
    records = SYNTHETIC
    print(f'{CORPUS} not found -- using the {len(records)}-record synthetic smoke corpus.')
    print('This proves the pipeline, not the vertical. Upload a real corpus before §6 matters.')

labels = sorted({l for r in records for l in r['output']['entities']})
print('\nlabels:', labels)
print(json.dumps(records[0], indent=2)[:300])

## 5. Dataset preflight — the two silent-loss checks

Neither of these raises inside GLiNER2. They quietly reduce what the model can learn, and
the natural response to the resulting bad metrics — train longer — makes it worse. So they
are measured here, on the cheap runtime, before a GPU session is spent.

- **Width:** `model.py:594-604` sets a label only when `0 <= width < max_width`. The bound is
  exclusive, so a span of exactly 8 tokens is already unlearnable. Long PII (addresses,
  clause text) is directly exposed.
- **Resolvability:** `sanitize()` drops the **entire entity type** for a record if any one of
  its mentions is not found verbatim (`data.py:756`, `:796`).

This mirrors `training/dataset_preflight.py`, which is the tested source of record; it is
inlined because Colab has no repo checkout.

In [ ]:
import re

# Verbatim copy of GLiNER2's WhitespaceTokenSplitter._PATTERN (processor.py:231-238).
# A naive text.lower().split() here produces FALSE POSITIVES: it leaves punctuation glued
# to the token ('inc.' vs 'inc'), so real, resolvable mentions get reported as missing and
# the preflight degrades into noise nobody reads. This pattern emits punctuation as its own
# token, which is what GLiNER2 actually matches against.
_SPLIT = re.compile(
    r"""(?:https?://[^\s]+|www\.[^\s]+)
    |[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}
    |@[a-z0-9_]+
    |\w+(?:[-_]\w+)*
    |\S""",
    re.VERBOSE | re.IGNORECASE,
)

def tokens(text):
    return [m.group().lower() for m in _SPLIT.finditer(text)]

def preflight(records, max_width=MAX_WIDTH):
    wide, unresolved = [], []
    for i, r in enumerate(records):
        haystack = tokens(r['input'])
        for label, mentions in r['output']['entities'].items():
            for m in mentions:
                needle = tokens(m)
                if len(needle) >= max_width:
                    wide.append((i, label, m))
                found = needle and any(haystack[j:j + len(needle)] == needle
                                       for j in range(len(haystack) - len(needle) + 1))
                if not found:
                    unresolved.append((i, label, m))
    return wide, unresolved

wide, unresolved = preflight(records)

print(f'spans >= {MAX_WIDTH} tokens (SILENTLY UNLEARNABLE): {len(wide)}')
for i, label, m in wide[:10]:
    print(f'  rec {i} [{label}] {m!r}')

print(f'\nunresolvable mentions (DROP WHOLE ENTITY TYPE for that record): {len(unresolved)}')
for i, label, m in unresolved[:10]:
    print(f'  rec {i} [{label}] {m!r}')

if wide:
    print('\nA vertical whose key entity is routinely wider than max_width needs a different')
    print('plan, not more epochs. Decide before training, not after reading bad metrics.')

---

**The contract is proven and the corpus is measured, having spent no GPU quota.** Everything
below needs the 1.2 GB base checkpoint and a GPU.

---

## 6. Train

Target modules exclude `count_pred` and `classifier`: on entity-only data the count-loss path
is not exercised (`model.py:392-395`), so they get **zero gradient** — LoRA on them is dead
weight, and `classifier` is not ported to the Rust runtime at all. `count_embed.gru` and
`pos_embedding` cannot be LoRA'd either, since PEFT targets `nn.Linear` only.

New labels are still learnable from the encoder alone: GLiNER2 has **no per-label embedding
table**. Label semantics are encoder hidden states read at `[E]` marker positions, and the
gather/scorer stages are parameter-free. That is exactly why a merge-only runtime works — the
loader can add deltas to existing weights, but could never add new head parameters.

In [ ]:
%pip install -q gliner2 huggingface_hub

import os
if HF_CACHE:
    # Cache the 1.2 GB base on Drive so a disconnect does not cost another download.
    os.makedirs(HF_CACHE, exist_ok=True)
    os.environ['HF_HOME'] = HF_CACHE

from huggingface_hub import snapshot_download
BASE_DIR = snapshot_download(BASE_REPO)
print('base dir:', BASE_DIR)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE -- this will be slow')

In [ ]:
import inspect
from gliner2.training.trainer import train_gliner2

kwargs = dict(
    # --- LoRA ---
    use_lora=True,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    lora_use_dora=False,          # DoRA emits lora_magnitude_vector keys -> parser rejects
    # GLiNER2 reads these as module GROUPS in its own apply_lora, not as PEFT suffixes.
    # Section 7 proves whether the resulting paths are mergeable; if it reports zero adapter
    # tensors or unmergeable paths, rerun with lora_target_modules=FQ_TARGETS.
    lora_target_modules=['encoder', 'span_rep'],   # NOT count_pred/classifier (no gradient)
    save_adapter_only=True,
    # --- dtype: adapter MUST end up F32 (lora.rs:127), so no mixed precision ---
    fp16=False,
    bf16=False,
    # --- data integrity ---
    synthetic_entity_label_prob=0.0,   # default 0.2 relabels to 'entity 1'/'entity 2'
    # --- optimisation ---
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    encoder_lr=ENCODER_LR,
    task_lr=TASK_LR,
)

# Every kwarg above is load-bearing: silently dropping one (a renamed parameter in a newer
# gliner2) yields a *successful* run producing an adapter the loader rejects. Fail here
# instead, while it costs nothing.
params = inspect.signature(train_gliner2).parameters
if not any(p.kind is inspect.Parameter.VAR_KEYWORD for p in params.values()):
    unknown = sorted(set(kwargs) - set(params))
    assert not unknown, (
        f'this gliner2 build does not accept {unknown}; the defaults those would have '
        f'overridden produce a rejected adapter. Accepted: {sorted(params)}'
    )

result = train_gliner2(model_path=BASE_DIR, train_data=records, output_dir=OUT_DIR, **kwargs)
print(result)

## 7. Trained-adapter key check

§2 proved PEFT's naming convention on a stub. This runs the same check against what
GLiNER2's own `apply_lora` actually produced, which is the part §2 could not cover: the *set*
of modules targeted. Every adapter module path must exactly equal some base key minus
`.weight`, or `merge_into_base` refuses the adapter (`lora.rs:246-253`).

In [ ]:
import glob, os

cand = glob.glob(f'{OUT_DIR}/**/adapter_model.safetensors', recursive=True)
assert cand, f'no adapter_model.safetensors under {OUT_DIR}; check save_adapter_only'
trained = sorted(cand, key=os.path.getmtime)[-1]
print('trained adapter:', trained)

with safe_open(trained, framework='pt') as f:
    adapter_keys = list(f.keys())
print('adapter tensors:', len(adapter_keys))
print('example:', adapter_keys[0] if adapter_keys else 'NONE')

SUFFIXES = ('.lora_A.weight', '.lora_B.weight')
bad_shape = [k for k in adapter_keys
             if not (k.startswith('base_model.model.') and k.endswith(SUFFIXES))]

paths = set()
for k in adapter_keys:
    s = k.removeprefix('base_model.model.')
    for suf in SUFFIXES:
        if s.endswith(suf):
            paths.add(s[:-len(suf)])

missing = sorted(p for p in paths if f'{p}.weight' not in set(BASE_KEYS))

print('\nkeys not in PEFT shape (rejected by parse_lora_key):', len(bad_shape))
for k in bad_shape[:10]:
    print('  ', k)
print('module paths with no exact base match (rejected by merge_into_base):', len(missing))
for p in missing[:10]:
    print('  ', p)

assert paths, 'adapter is empty; lora_target_modules matched nothing -- retry with FQ_TARGETS'
assert not bad_shape, 'adapter contains keys the strict parser will reject'
assert not missing, 'adapter targets modules absent from the base checkpoint'
print(f'\nOK: {len(paths)} module paths all match the base checkpoint exactly.')

## 8. Export

Uses the repo's canonical `training/adapter_export.py` rather than reimplementing the
contract here, so the notebook cannot drift from the tested implementation. Upload that file
once; it is cached to Drive for later sessions.

In [ ]:
import importlib.util, shutil

exporter = WORK / 'adapter_export.py'
if not exporter.exists():
    print('Upload training/adapter_export.py from the hacienda-engine repo:')
    from google.colab import files
    up = files.upload()
    exporter.write_bytes(next(iter(up.values())))

spec = importlib.util.spec_from_file_location('adapter_export', str(exporter))
adapter_export = importlib.util.module_from_spec(spec)
spec.loader.exec_module(adapter_export)
print('loaded canonical exporter from', exporter)

In [ ]:
import numpy as np

with safe_open(trained, framework='pt') as f:
    tensors = {k: f.get_tensor(k).detach().cpu().float().numpy() for k in f.keys()}

# base_model_dir is passed only so the exporter can derive the deployed basename for
# base_model_name_or_path -- it is not read from disk.
out = pathlib.Path(adapter_export.write_adapter(
    ADAPTER_DIR,
    tensors,
    base_model_dir=f'/models/{DEPLOY_BASENAME}',
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    base_tensor_keys=BASE_KEYS,   # runs the merge coverage check again, at export
))

cfg = json.loads((out / 'adapter_config.json').read_text())
print(json.dumps(cfg, indent=2))

with safe_open(str(out / 'adapter_model.safetensors'), framework='numpy') as f:
    dts = {f.get_tensor(k).dtype for k in f.keys()}
print('exported dtypes:', dts)
assert dts == {np.dtype('float32')}, 'adapter must be F32'
assert cfg['base_model_name_or_path'] == DEPLOY_BASENAME
print(f'\nOK: F32, base_model_name_or_path pinned to the deployed basename.')
print('adapter written to', out, '(on Drive)' if USE_DRIVE else '(EPHEMERAL -- download it now)')

In [ ]:
zip_path = f'/content/adapter-{VERTICAL_ID}.zip'
shutil.make_archive(zip_path[:-4], 'zip', root_dir=str(out))
print(zip_path, os.path.getsize(zip_path), 'bytes')

from google.colab import files
files.download(zip_path)

## 9. Next: verify the merge locally

The adapter is a few MB — small enough to pull down and test against a real base. Unzip it
beside an **F32** base and run Task 4 of the plan:

```bash
cargo test -p hacienda-core --features ner-candle --test lora_adapter_contract
```

then the `#[ignore]`d merge smoke test asserting `from_candle_local(f32_base, Some(adapter))`
returns `Ok` **and** changes output versus the base model. Loading is necessary but not
sufficient — an adapter that merges cleanly and changes nothing is a failed run that looks
like a successful one.

Do not merge against `apps/hacienda-studio/public/models/gliner2/model.safetensors` — that
file is F16, the derived distribution artifact from `scripts/convert_gliner2_f16.py`, and
cannot be a merge base.

**Shipping a vertical adapter remains gated** on Tier 1 calibration measurably failing
(`2026-07-31-vertical-model-specialisation-design.md` §8). This notebook produces the
evidence for that gate; it does not open it.